In [ ]:
# COMPLETE AI AGENT USING LANGGRAPH

# No OpenAI API Key Required

In [ ]:
# 1. INSTALL REQUIRED LIBRARIES

!pip install -q -U langgraph langchain langchain-core \
    langchain-huggingface transformers accelerate

In [ ]:
# 2. IMPORT LIBRARIES

from typing import TypedDict  #english
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)
from langchain_huggingface import HuggingFacePipeline

In [ ]:
# 3. LOAD HUGGING FACE MODEL

print("Loading AI model...")

model_name = "Qwen/Qwen2.5-0.5B-Instruct"   #quen

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.2,
    do_sample=True
)

llm = HuggingFacePipeline(
    pipeline=pipe
)

print("AI model loaded successfully!")

Loading AI model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

AI model loaded successfully!


In [ ]:
# 4. CREATE TOOLS
@tool
def calculator(expression: str) -> str:
    """
    Performs mathematical calculations.
    Example: 25 * 10
    """

    try:
        # Allow only basic mathematical characters
        allowed = "0123456789+-*/(). "

        if not all(char in allowed for char in expression):
            return "Invalid mathematical expression."

        result = eval(expression)

        return str(result)

    except Exception:
        return "Could not calculate the expression."


@tool
def college_information(question: str) -> str:
    """
    Provides information about college.
    """

    question = question.lower()

    if "attendance" in question:
        return "The minimum attendance required is 75%."

    elif "timing" in question or "time" in question:
        return "College timing is from 9 AM to 4 PM."

    elif "library" in question:
        return "The library is open from 8 AM to 6 PM."

    elif "exam" in question:
        return "The semester examination is conducted according to the college academic schedule."

    elif "placement" in question:
        return "The placement cell helps students with training, interviews and placement opportunities."

    else:
        return "Sorry, I don't have information about that."


In [ ]:
# 5. DEFINE STATE

class AgentState(TypedDict):
    question: str
    answer: str

In [ ]:
# 6. CREATE AI AGENT

def ai_agent(state: AgentState):

    question = state["question"]
    question_lower = question.lower()

    # TOOL 1: CALCULATOR

    math_words = [
        "+",
        "-",
        "*",
        "/",
        "calculate",
        "multiply",
        "divide",
        "add",
        "subtract"
    ]

    if any(word in question_lower for word in math_words):

        # Handle common natural language questions

        expression = question_lower

        expression = expression.replace("what is", "")
        expression = expression.replace("calculate", "")
        expression = expression.replace("multiply", "")
        expression = expression.replace("divide", "")
        expression = expression.replace("add", "")
        expression = expression.replace("subtract", "")
        expression = expression.replace("by", "*")
        expression = expression.replace("x", "*")

        # Keep only mathematical characters

        expression = "".join(
            char for char in expression
            if char in "0123456789+-*/(). "
        )

        if expression.strip():
            result = calculator.invoke(expression)

            return {
                "question": question,
                "answer": f"The calculated answer is {result}"
            }

    # TOOL 2: COLLEGE INFORMATION

    college_words = [
        "college",
        "attendance",
        "library",
        "placement",
        "exam",
        "timing"
    ]

    if any(word in question_lower for word in college_words):

        result = college_information.invoke(question)

        return {
            "question": question,
            "answer": result
        }

    # TOOL 3: GENERAL AI QUESTION

    prompt = f"""
    You are a helpful AI assistant.
    Answer the following question clearly.

    Question: {question}
    """

    response = llm.invoke(prompt)

    return {
        "question": question,
        "answer": response
    }


# TEST THE AI AGENT

result = ai_agent({
    "question": "What is LangChain?"
})

print(result)


{'question': 'What is LangChain?', 'answer': '\n    You are a helpful AI assistant.\n    Answer the following question clearly.\n\n    Question: What is LangChain?\n     - A collection of open-source machine learning libraries\n     - A language model for natural language processing\n     - A framework for building chatbots and virtual assistants\n     - A platform for developing web applications\nAnswer:\n\nLangChain is a collection of open-source machine learning libraries. It provides a wide range of tools and techniques for building machine learning models, including natural language processing (NLP) and computer vision. The library includes various frameworks and tools that can be used to develop chatbots and virtual assistants, as well as other types of applications. It also supports the development of web applications using Python. Therefore, among the given options, "A collection of open-source machine learning libraries" is the most accurate description of what LangChain is. H

In [ ]:
# 7. BUILD LANGGRAPH

graph = StateGraph(AgentState)

# Add agent node
graph.add_node(
    "ai_agent",
    ai_agent
)

# Starting point
graph.add_edge(
    START,
    "ai_agent"
)

# Ending point
graph.add_edge(
    "ai_agent",
    END
)

In [ ]:
# 8. COMPILE GRAPH

app = graph.compile()

print("\nLangGraph Agent created successfully!")


LangGraph Agent created successfully!


In [ ]:
# 9. CHAT WITH THE AI AGENT

print("\n============================================")
print("       LANGGRAPH AI AGENT")
print("============================================")

print("\nYou can ask questions like:")

print("1. What is 25 * 10?")
print("2. What is the attendance requirement?")
print("3. What are the college timings?")
print("4. Is the library open?")
print("5. Tell me about placements.")

print("\nType 'exit' to stop.\n")


       LANGGRAPH AI AGENT

You can ask questions like:
1. What is 25 * 10?
2. What is the attendance requirement?
3. What are the college timings?
4. Is the library open?
5. Tell me about placements.

Type 'exit' to stop.



In [ ]:
# 10. CONTINUOUS CHAT

while True:

    question = input("You: ")

    if question.lower() == "exit":

        print("\nAI Agent: Goodbye! 👋")
        break

    if question.strip() == "":
        print("Please enter a question.")
        continue

    result = app.invoke({
        "question": question,
        "answer": ""
    })

    print("\nAI Agent:", result["answer"])
    print()

You: What is 25 * 10?

AI Agent: The calculated answer is 250

You: exit

AI Agent: Goodbye! 👋
